In [6]:
import os
import numpy as np
import pandas as pd

DATA_PATH = '../data/raw/accepted_2007_to_2018Q4.csv'

In [7]:
cols = ['issue_d', 'loan_status', 'loan_amnt', 'int_rate', 'annual_inc',
        'dti', 'fico_range_low', 'term', 'grade', 'emp_length',
        'home_ownership', 'purpose']

chunks = []
chunk_size = 100_000

for chunk in pd.read_csv(DATA_PATH, usecols=cols, chunksize=chunk_size, low_memory=False):
    chunk['issue_d'] = pd.to_datetime(chunk['issue_d'], format='%b-%Y', errors='coerce')
    filtered = chunk[chunk['issue_d'].dt.year.isin([2015, 2016])]
    chunks.append(filtered)

df = pd.concat(chunks, ignore_index=True)
print(df.shape)

(855502, 12)


In [8]:
ref_window = df[df['issue_d'].dt.year == 2015]
later_window = df[df['issue_d'].dt.year == 2016]
print(ref_window.shape, later_window.shape)

(421095, 12) (434407, 12)


In [9]:
print(ref_window['loan_status'].value_counts())
print(later_window['loan_status'].value_counts())

loan_status
Fully Paid            299742
Charged Off            75803
Current                43299
Late (31-120 days)      1359
In Grace Period          612
Late (16-30 days)        279
Default                    1
Name: count, dtype: int64
loan_status
Fully Paid            224853
Current               134061
Charged Off            68242
Late (31-120 days)      4546
In Grace Period         1793
Late (16-30 days)        902
Default                   10
Name: count, dtype: int64


Note: 2016 has a much higher share of Current loans (~31%) than 2015 (~10%), because 2016 loans have had less time to mature. Keep this in mind as a limitation.

In [10]:
ref_resolved = ref_window[ref_window['loan_status'].isin(['Fully Paid', 'Charged Off'])].copy()
later_resolved = later_window[later_window['loan_status'].isin(['Fully Paid', 'Charged Off'])].copy()
print(ref_resolved.shape, later_resolved.shape)

(375545, 12) (293095, 12)


In [11]:
ref_resolved['default'] = (ref_resolved['loan_status'] == 'Charged Off').astype(int)
later_resolved['default'] = (later_resolved['loan_status'] == 'Charged Off').astype(int)
print(ref_resolved['default'].mean(), later_resolved['default'].mean())

0.20184798093437537 0.2328323581091455


In [12]:
feature_cols = ['loan_amnt', 'int_rate', 'annual_inc', 'dti', 'fico_range_low',
                 'term', 'grade', 'emp_length', 'home_ownership', 'purpose']

print(ref_resolved[feature_cols].isnull().mean())
print(later_resolved[feature_cols].isnull().mean())

loan_amnt         0.000000
int_rate          0.000000
annual_inc        0.000000
dti               0.000005
fico_range_low    0.000000
term              0.000000
grade             0.000000
emp_length        0.058736
home_ownership    0.000000
purpose           0.000000
dtype: float64
loan_amnt         0.000000
int_rate          0.000000
annual_inc        0.000000
dti               0.000130
fico_range_low    0.000000
term              0.000000
grade             0.000000
emp_length        0.064409
home_ownership    0.000000
purpose           0.000000
dtype: float64


In [13]:
# dti: trivial amount missing, safe to drop those rows
ref_resolved = ref_resolved.dropna(subset=['dti'])
later_resolved = later_resolved.dropna(subset=['dti'])

# emp_length: ~6% missing, treat as its own category rather than losing rows
ref_resolved['emp_length'] = ref_resolved['emp_length'].fillna('Unknown')
later_resolved['emp_length'] = later_resolved['emp_length'].fillna('Unknown')

In [14]:
pd.set_option('display.max_columns', None)
numeric_cols = ['loan_amnt', 'int_rate', 'annual_inc', 'dti', 'fico_range_low']

print(ref_resolved[numeric_cols].describe())
print(later_resolved[numeric_cols].describe())

           loan_amnt       int_rate    annual_inc            dti  \
count  375543.000000  375543.000000  3.755430e+05  375543.000000   
mean    14641.578860      12.385171  7.626962e+04      18.942635   
std      8522.696364       4.315012  7.499088e+04       8.887976   
min      1000.000000       5.320000  1.200000e+03       0.000000   
25%      8000.000000       9.170000  4.500000e+04      12.390000   
50%     12550.000000      12.290000  6.500000e+04      18.380000   
75%     20000.000000      14.990000  9.000000e+04      25.100000   
max     35000.000000      28.990000  9.500000e+06     999.000000   

       fico_range_low  
count   375543.000000  
mean       693.517533  
std         30.335932  
min        660.000000  
25%        670.000000  
50%        685.000000  
75%        710.000000  
max        845.000000  
           loan_amnt       int_rate    annual_inc            dti  \
count  293057.000000  293057.000000  2.930570e+05  293057.000000   
mean    14467.138304      13.086399

In [15]:
for col in ['term', 'grade', 'emp_length', 'home_ownership', 'purpose']:
    print(col)
    print(ref_resolved[col].value_counts(dropna=False))
    print()

term
term
36 months    283024
60 months     92519
Name: count, dtype: int64

grade
grade
B    106916
C    104615
A     71890
D     53479
E     28535
F      8212
G      1896
Name: count, dtype: int64

emp_length
emp_length
10+ years    123990
2 years       33758
< 1 year      30938
3 years       30031
1 year        24885
5 years       22344
Unknown       22056
4 years       22034
8 years       19257
7 years       16423
6 years       15000
9 years       14827
Name: count, dtype: int64

home_ownership
home_ownership
MORTGAGE    182128
RENT        152342
OWN          41071
ANY              2
Name: count, dtype: int64

purpose
purpose
debt_consolidation    222049
credit_card            90091
home_improvement       22707
other                  17944
major_purchase          6730
medical                 3684
car                     3203
small_business          3081
moving                  2313
vacation                2191
house                   1331
renewable_energy         214
wedding       

In [17]:
os.makedirs("../data/processed", exist_ok=True)
ref_resolved.to_parquet("../data/processed/ref_2015.parquet")
later_resolved.to_parquet("../data/processed/later_2016.parquet")